In [3]:
"""
Regression test for the schrieffer_wolff_correction bug:
  E_X = self.fock_states @ self.fock_energies   (WRONG: shape mismatch,
        and conceptually wrong even when shapes happened to align, since
        fock_energies is already a per-state vector, not a per-mode one
        to be matrix-multiplied against fock_states)
fixed to:
  E_X = self.fock_energies                       (fock_energies IS the
        per-state energy vector already -- diagonalized-subspace
        eigenvalues when a Kerr term is present, plain diag_vals
        otherwise -- build_HC_fock always returns it pre-computed)

Checks:
  1. No-Kerr boson case matches ssh_chain_single.py exactly (this would
     have crashed before the fix, confirming the bug was not Kerr-specific).
  2. Kerr-active boson case (where the bug was originally reported) now
     produces a finite, Hermitian H_SW instead of crashing.
  3. Fermion case (no Kerr supported) still works.
"""
import numpy as np
import scipy.sparse as sp

from ssh_chain_single import PHOTONICChain as Orig
from ssh_chain_sq import PHOTONICChain as SQ


def to_dense(M):
    return M.toarray() if sp.issparse(M) else np.asarray(M)


def cmp(a, b, name, atol=1e-8):
    a, b = to_dense(a), to_dense(b)
    d = np.max(np.abs(a - b))
    ok = np.allclose(a, b, atol=atol)
    print(f"  {name}: max diff {d:.3e}  match={ok}")
    return ok


def test_no_kerr_matches_original():
    print("--- no-Kerr boson vs ssh_chain_single ---")
    N, Omega_C, Chi_C, omega_r, gamma, fock_photon, Omega_J, Chi_J, PBC = \
        4, 0.3, 0.5, 0.13, 0.02, 4, 1.0, 0.0, False
    n_exc = 2

    orig = Orig(N=N, Omega_C=Omega_C, Chi_C=Chi_C, omega_r=omega_r, gamma=gamma,
                fock_photon=fock_photon, Omega_J=Omega_J, Chi_J=Chi_J, PBC=PBC)
    orig.matter_hamiltonians(n_excitations=n_exc)
    orig.build_full_hamiltonian()
    orig.schrieffer_wolff_correction()

    sq = SQ(N=N, Omega_C=Omega_C, Chi_C=Chi_C, omega_r=omega_r, gamma=gamma,
            fock_photon=fock_photon, Omega_J=Omega_J, Chi_J=Chi_J,
            kerr=None, PBC=PBC, statistics='boson')
    sq.matter_hamiltonians(n_excitations=n_exc)
    sq.build_full_hamiltonian()
    sq.schrieffer_wolff_correction()

    ok = cmp(orig.H_C_fock, sq.H_C_fock, "H_C_fock")
    ok &= cmp(orig.H_J_fock, sq.H_J_fock, "H_J_fock")
    ok &= cmp(orig.H_SW, sq.H_SW, "H_SW")
    ok &= cmp(orig.H, sq.H, "full H")
    return ok


def _sanity_check(label, **kwargs):
    chain = SQ(**kwargs)
    chain.matter_hamiltonians(n_excitations=3)
    chain.build_full_hamiltonian()
    chain.schrieffer_wolff_correction()
    H_SW = to_dense(chain.H_SW)
    herm = np.allclose(H_SW, H_SW.conj().T, atol=1e-8)
    finite = np.all(np.isfinite(H_SW))
    print(f"  {label}: hermitian={herm} finite={finite} max_abs={np.max(np.abs(H_SW)):.4e}")
    return herm and finite


def test_kerr_boson_sane():
    print("--- Kerr-active boson sanity ---")
    ok = _sanity_check("kerr=0.5", N=4, Omega_C=0.3, Chi_C=0.5, omega_r=0.13, gamma=0.02,
                        fock_photon=4, Omega_J=1.0, Chi_J=0.0, kerr=0.5, PBC=False, statistics='boson')
    ok &= _sanity_check("kerr=1.0, different params", N=3, Omega_C=0.5, Chi_C=0.2, omega_r=0.5,
                         gamma=0.01, fock_photon=5, Omega_J=0.3, Chi_J=0.4, kerr=1.0, PBC=False,
                         statistics='boson')
    return ok


def test_fermion_sane():
    print("--- fermion sanity (no Kerr) ---")
    return _sanity_check("fermion", N=4, Omega_C=0.3, Chi_C=0.5, omega_r=0.13, gamma=0.02,
                          fock_photon=4, Omega_J=1.0, Chi_J=0.5, kerr=None, PBC=False,
                          statistics='fermion')


if __name__ == "__main__":
    results = [
        test_no_kerr_matches_original(),
        test_kerr_boson_sane(),
        test_fermion_sane(),
    ]
    print()
    print("ALL PASS" if all(results) else "SOME FAILED")

--- no-Kerr boson vs ssh_chain_single ---
  H_C_fock: max diff 0.000e+00  match=True
  H_J_fock: max diff 2.220e-16  match=True
  H_SW: max diff 0.000e+00  match=True
  full H: max diff 1.388e-17  match=True
--- Kerr-active boson sanity ---
  kerr=0.5: hermitian=True finite=True max_abs=4.6501e+00
  kerr=1.0, different params: hermitian=True finite=True max_abs=5.7462e-02
--- fermion sanity (no Kerr) ---
  fermion: hermitian=True finite=True max_abs=4.9199e-03

ALL PASS
